# PhasePhyto: Physics-Informed Differentiable Phase Congruency for Cross-Domain Botanical Visual Recognition

This notebook implements the complete **PhasePhyto** framework -- a three-stream hybrid architecture that fuses:

1. **Phase Congruency Stream** -- 24 Log-Gabor filters (6 orientations x 4 scales) extract illumination-invariant structural maps
2. **ViT Semantic Backbone** -- `vit_base_patch16_224` extracts high-level semantic features  
3. **Illumination-Normalised Stream** -- CIELAB + CLAHE on luminance channel

These are fused via **cross-attention** where structural tokens (Q) attend to semantic tokens (K,V).

**Key property**: `PC(image) = PC(image * k)` for any `k > 0` -- mathematical illumination invariance.

---

**Use Case**: Plant Disease Detection (PlantVillage -> PlantDoc lab-to-field domain shift)

**Runtime**: Colab T4 GPU (~15GB VRAM)

## 0. Setup & Installation

In [ ]:
# Install dependencies
# Colab already includes torch/torchvision/Pillow in the standard GPU runtime.
# Install the project-specific packages used by training, evaluation, and plots.
!pip install -q timm opencv-python-headless scikit-learn matplotlib seaborn tqdm pyyaml


In [ ]:
import os
import json
import math
import random
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import timm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Reproducibility
SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

## 1. Configuration

In [ ]:
# ============================================================
# CONFIGURATION - Edit these to match your setup
# ============================================================

CONFIG = {
    # Model
    "backbone_name": "vit_base_patch16_224",  # ViT-B/16 backbone
    "fusion_dim": 256,
    "pc_scales": 4,
    "pc_orientations": 6,
    "num_heads": 4,
    "dropout": 0.1,
    "image_size": 224,

    # Training
    "lr": 5e-5,
    "weight_decay": 1e-2,
    "epochs": 20,
    "warmup_epochs": 3,
    "batch_size": 16,  # Reduce to 8 if OOM on T4
    "grad_clip": 1.0,
    "patience": 8,

    # Data/import paths
    "use_synthetic": True,  # Set False when using real PlantVillage/PlantDoc
    "data_root": "/content/data",  # Fast local/SSD working data root
    "plantvillage_dir": "",  # Optional direct source path, e.g. mounted Drive/SSD
    "plantdoc_dir": "",      # Optional direct target path, e.g. mounted Drive/SSD
    "num_workers": 2,

    # Fast real-data hydration. Recommended for Colab: keep master data/tar files
    # in Drive, extract archives into /content/data, then train from local SSD.
    "hydrate_local_data_from_archives": True,
    "force_rehydrate_local_data": False,
    "local_data_root": "/content/data",
    "drive_archive_dir": "/content/drive/MyDrive/PhasePhyto/data/archives",

    # Pipeline/storage backend
    # Options:
    #   "drive"        -> save all artifacts directly to Google Drive
    #   "colab_ssd"    -> save to Colab local SSD (/content), fastest but ephemeral
    #   "external_ssd" -> save to a mounted/hooked SSD path you provide below
    "storage_backend": "drive",
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "colab_project_dir": "/content/phasephyto_runs",
    "external_ssd_project_dir": "/content/ssd/PhasePhyto",  # change to your mounted SSD path
    "run_name": None,  # None => auto timestamped run folder
    "download_outputs_at_end": False,  # Drive/SSD storage is preferred
}

print("Configuration loaded.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 1.1 Pipeline Segmentation & Storage Routing

All generated artifacts are written under one run directory. Set
`CONFIG["storage_backend"]` to `"drive"`, `"colab_ssd"`, or `"external_ssd"`
to route checkpoints, plots, metrics, and manifests to Google Drive, fast local
Colab storage, or a mounted/hooked SSD path.


In [ ]:
# ============================================================
# PIPELINE SEGMENTATION + STORAGE-BACKEND ARTIFACT ROUTING
# ============================================================
from datetime import datetime

backend = CONFIG.get("storage_backend", "drive").lower()

if backend == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        OUTPUT_ROOT = Path(CONFIG["drive_project_dir"])
        print(f"Google Drive mounted. Output root: {OUTPUT_ROOT}")
    except Exception as exc:
        print(f"Drive mount failed ({exc}); falling back to Colab local SSD.")
        backend = "colab_ssd"
        OUTPUT_ROOT = Path(CONFIG["colab_project_dir"])
elif backend == "external_ssd":
    OUTPUT_ROOT = Path(CONFIG["external_ssd_project_dir"])
    if not OUTPUT_ROOT.parent.exists():
        print(f"WARNING: external SSD parent does not exist yet: {OUTPUT_ROOT.parent}")
    print(f"Using external/hooked SSD output root: {OUTPUT_ROOT}")
elif backend == "colab_ssd":
    OUTPUT_ROOT = Path(CONFIG["colab_project_dir"])
    print(f"Using Colab local SSD output root: {OUTPUT_ROOT}")
else:
    raise ValueError(f"Unknown storage_backend={backend!r}; use drive, colab_ssd, or external_ssd")

RUN_NAME = CONFIG["run_name"] or datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
MANIFEST_PATH = RUN_DIR / "run_manifest.json"

for directory in [CHECKPOINT_DIR, PLOTS_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ARTIFACTS = {
    "storage_backend": backend,
    "run_dir": str(RUN_DIR),
    "checkpoints": str(CHECKPOINT_DIR),
    "plots": str(PLOTS_DIR),
    "results": str(RESULTS_DIR),
}

print("Pipeline stages:")
for stage in [
    "0 setup + storage routing",
    "1 data import / synthetic generation",
    "2 model build + physics checks",
    "3 PhasePhyto training",
    "4 source/target evaluation",
    "5 baseline training",
    "6 comparison + visualizations",
    "7 artifact manifest",
]:
    print(f"  - {stage}")

print("\nArtifact directories:")
for name, value in ARTIFACTS.items():
    print(f"  {name}: {value}")


## 1.2 Hydrate Local Data From Drive Tar Archives

For real training, avoid reading thousands of image files directly from Google Drive. If the downloader notebook created `plantvillage.tar` and `plantdoc.tar`, this cell extracts them into `/content/data` and points training at the fast local Colab SSD.


In [ ]:
# ============================================================
# HYDRATE FAST LOCAL DATA CACHE FROM DRIVE TAR ARCHIVES
# ============================================================

import subprocess

if (not CONFIG["use_synthetic"]) and CONFIG.get("hydrate_local_data_from_archives", False):
    local_data_root = Path(CONFIG["local_data_root"])
    archive_dir = Path(CONFIG["drive_archive_dir"])
    local_data_root.mkdir(parents=True, exist_ok=True)

    for dataset_name in ["plantvillage", "plantdoc"]:
        archive_path = archive_dir / f"{dataset_name}.tar"
        target_dir = local_data_root / dataset_name

        if target_dir.exists() and any(target_dir.iterdir()) and not CONFIG["force_rehydrate_local_data"]:
            print(f"Local {dataset_name} already exists at {target_dir}; skipping extraction")
        else:
            if not archive_path.exists():
                raise FileNotFoundError(
                    f"Missing archive: {archive_path}. Run PhasePhyto_Download_Data_To_Drive.ipynb "
                    "with create_tar_archives=True first, or set plantvillage_dir/plantdoc_dir manually."
                )
            if target_dir.exists() and CONFIG["force_rehydrate_local_data"]:
                import shutil
                shutil.rmtree(target_dir)
            print(f"Extracting {archive_path} -> {local_data_root}")
            subprocess.run(["tar", "-xf", str(archive_path), "-C", str(local_data_root)], check=True)

    CONFIG["plantvillage_dir"] = str(local_data_root / "plantvillage")
    CONFIG["plantdoc_dir"] = str(local_data_root / "plantdoc")
    print("Training data hydrated to local Colab SSD:")
    print(f"  PlantVillage: {CONFIG['plantvillage_dir']}")
    print(f"  PlantDoc:      {CONFIG['plantdoc_dir']}")
else:
    print("Skipping local archive hydration (synthetic mode or hydrate_local_data_from_archives=False).")


## 2. Data: PlantVillage & PlantDoc

**PlantVillage** (source domain): 54,305 lab images, 38 classes  
**PlantDoc** (target domain): 2,598 field images, 27 classes  

We train on PlantVillage and evaluate zero-shot on PlantDoc.

In [ ]:
# Download PlantVillage dataset from Kaggle
# Option A: Upload your kaggle.json and use the Kaggle API
# Option B: Mount Google Drive with pre-downloaded data
# Option C: Use synthetic data to verify the full training/evaluation pipeline

DATA_ROOT = Path(CONFIG["data_root"])
DATA_ROOT.mkdir(parents=True, exist_ok=True)

PLANTVILLAGE_DIR = (
    Path(CONFIG["plantvillage_dir"])
    if CONFIG.get("plantvillage_dir")
    else DATA_ROOT / "plantvillage"
)
PLANTDOC_DIR = (
    Path(CONFIG["plantdoc_dir"])
    if CONFIG.get("plantdoc_dir")
    else DATA_ROOT / "plantdoc"
)

# If PlantDoc is organised as plantdoc/test/<class> (as in the local repo helper),
# the dataset cell below will automatically resolve PLANTDOC_DIR -> PLANTDOC_DIR/test.

# --- Option A: Kaggle API ---
# Uncomment the following lines if you have kaggle.json uploaded:
#
# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p {DATA_ROOT}
# !unzip -q {DATA_ROOT}/plantvillage-dataset.zip -d {DATA_ROOT}/plantvillage_raw
# Then organise into PLANTVILLAGE_DIR with class subdirectories.

# --- Option B: Mount Google Drive or an external/hooked SSD with pre-downloaded data ---
# from google.colab import drive
# drive.mount('/content/drive')
# PLANTVILLAGE_DIR = Path('/content/drive/MyDrive/datasets/plantvillage')
# PLANTDOC_DIR = Path('/content/drive/MyDrive/datasets/plantdoc')
# Or set CONFIG['plantvillage_dir'] / CONFIG['plantdoc_dir'] to mounted SSD paths.

# --- Option C: Synthetic data for testing the pipeline ---
# Creates a small dummy dataset to verify that training, validation, target
# evaluation, baseline comparison, visualisation, and artifact download work.
USE_SYNTHETIC = CONFIG["use_synthetic"]  # Set in CONFIG

if USE_SYNTHETIC:
    print("Creating synthetic test data...")
    NUM_CLASSES = 10
    SAMPLES_PER_CLASS = 50
    rng = np.random.default_rng(SEED)

    for split_name, split_dir, brightness_range in [
        ("plantvillage", PLANTVILLAGE_DIR, (0.8, 1.2)),
        ("plantdoc", PLANTDOC_DIR, (0.3, 2.5)),
    ]:
        for c in range(NUM_CLASSES):
            class_dir = split_dir / f"class_{c:02d}"
            class_dir.mkdir(parents=True, exist_ok=True)
            n = SAMPLES_PER_CLASS if split_name == "plantvillage" else SAMPLES_PER_CLASS // 5
            for i in range(n):
                img = rng.integers(50, 200, (256, 256, 3), dtype=np.uint8)
                freq = (c + 1) * 3
                x_grid = np.linspace(0, freq * np.pi, 256)
                y_grid = np.linspace(0, freq * np.pi, 256)
                xx, yy = np.meshgrid(x_grid, y_grid)
                pattern = ((np.sin(xx + c) * np.cos(yy) + 1) * 40).astype(np.uint8)
                channel = np.clip(img[:, :, c % 3].astype(int) + pattern, 0, 255)
                img[:, :, c % 3] = channel.astype(np.uint8)
                brightness = rng.uniform(*brightness_range)
                img = np.clip(img.astype(float) * brightness, 0, 255).astype(np.uint8)
                Image.fromarray(img).save(class_dir / f"{split_name}_{c:02d}_{i:04d}.png")

    print(f"Synthetic data created: {PLANTVILLAGE_DIR} and {PLANTDOC_DIR}")
else:
    print("Using real data from:")
    print(f"  PlantVillage: {PLANTVILLAGE_DIR}")
    print(f"  PlantDoc: {PLANTDOC_DIR}")


In [ ]:
# ============================================================
# CLAHE Transform & Dual Transform Pipeline
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class CLAHETransform:
    """Apply CLAHE to L channel of CIELAB, preserving A/B colour vectors."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, image):
        if not isinstance(image, np.ndarray):
            image = np.array(image)
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l_ch, a_ch, b_ch = cv2.split(lab)
        l_ch = self.clahe.apply(l_ch)
        return cv2.cvtColor(cv2.merge([l_ch, a_ch, b_ch]), cv2.COLOR_LAB2RGB)


class DualTransform:
    """Returns (rgb_tensor, clahe_tensor) for both PhasePhyto streams."""
    def __init__(self, base_transform, normalize, clahe_fn=None):
        self.base = base_transform
        self.normalize = normalize
        self.clahe_fn = clahe_fn or CLAHETransform()
        self.to_tensor = transforms.ToTensor()

    def __call__(self, image):
        augmented = self.base(image)
        aug_np = np.array(augmented)
        clahe_np = self.clahe_fn(aug_np)
        rgb_tensor = self.normalize(self.to_tensor(aug_np))
        clahe_tensor = self.normalize(self.to_tensor(clahe_np))
        return rgb_tensor, clahe_tensor


def get_train_transforms(image_size=224):
    spatial = transforms.Compose([
        transforms.Resize(int(image_size * 1.15)),
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    ])
    return DualTransform(spatial, transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))


def get_val_transforms(image_size=224):
    spatial = transforms.Compose([
        transforms.Resize(int(image_size * 1.15)),
        transforms.CenterCrop(image_size),
    ])
    return DualTransform(spatial, transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))


print("Transforms defined.")

In [ ]:
# ============================================================
# Dataset Class + Split Resolution
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def has_image_files(path):
    path = Path(path)
    return path.exists() and any(
        child.is_file() and child.suffix.lower() in IMAGE_EXTENSIONS
        for child in path.iterdir()
    )


def has_direct_class_images(path):
    path = Path(path)
    return path.exists() and any(
        child.is_dir() and has_image_files(child)
        for child in path.iterdir()
    )


def resolve_image_folder(root, preferred_splits=("test", "val", "valid", "validation", "train")):
    """Resolve either a direct class root or a parent containing train/test splits."""
    root = Path(root)
    if has_direct_class_images(root):
        return root
    for split in preferred_splits:
        candidate = root / split
        if has_direct_class_images(candidate):
            return candidate
    return root


class PlantDiseaseDataset(Dataset):
    """Image folder dataset returning (rgb, clahe, label)."""

    def __init__(self, root, transform=None, class_to_idx=None):
        self.root = resolve_image_folder(root)
        self.transform = transform

        class_dirs = sorted([d for d in self.root.iterdir() if d.is_dir()])
        self.class_to_idx = class_to_idx or {d.name: i for i, d in enumerate(class_dirs)}
        self.classes = list(self.class_to_idx.keys())
        self.num_classes = len(self.classes)

        self.samples = []
        for cd in class_dirs:
            if cd.name not in self.class_to_idx:
                continue
            label = self.class_to_idx[cd.name]
            for img_path in sorted(cd.glob("*")):
                if img_path.suffix.lower() in IMAGE_EXTENSIONS:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            result = self.transform(image)
            if isinstance(result, tuple):
                return (*result, label)
            return result, label
        return image, label


class TransformSubset(Dataset):
    """Wraps a torch Subset to override the parent dataset's transform.

    CRITICAL FIX: random_split returns Subset objects that inherit the parent's
    transform. Without this wrapper, val_subset would use TRAINING augmentations
    (random crops, flips, color jitter), inflating validation metrics.
    """
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        self.dataset = subset.dataset
        self.classes = subset.dataset.classes
        self.num_classes = subset.dataset.num_classes

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        real_idx = self.subset.indices[idx]
        path, label = self.subset.dataset.samples[real_idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            result = self.transform(image)
            if isinstance(result, tuple):
                return (*result, label)
            return result, label
        return image, label


# Create datasets
train_tf = get_train_transforms(CONFIG["image_size"])
val_tf = get_val_transforms(CONFIG["image_size"])

source_root = resolve_image_folder(PLANTVILLAGE_DIR, ("train", "training"))
target_root = resolve_image_folder(PLANTDOC_DIR, ("test", "val", "valid", "validation", "train"))

train_dataset = PlantDiseaseDataset(source_root, transform=train_tf)
target_dataset = PlantDiseaseDataset(
    target_root, transform=val_tf, class_to_idx=train_dataset.class_to_idx
)

if len(train_dataset) == 0:
    raise RuntimeError(f"No source samples found under {source_root}")
if len(target_dataset) == 0:
    print(f"WARNING: no target samples matched source classes under {target_root}")

# Train/val split from PlantVillage/source only.
# FIX: PlantDoc/target is test-only and is not used for early stopping.
# FIX: Wrap val subset with TransformSubset to apply val transforms (no augmentation).
from torch.utils.data import random_split
train_size = int(0.85 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_split, val_split = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_subset = train_split                       # keeps train augmentations
val_subset = TransformSubset(val_split, val_tf)  # deterministic val transforms

NUM_CLASSES = train_dataset.num_classes

train_loader = DataLoader(
    train_subset, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], pin_memory=(DEVICE == "cuda"), drop_last=True
)
val_loader = DataLoader(
    val_subset, batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"], pin_memory=(DEVICE == "cuda")
)
target_loader = DataLoader(
    target_dataset, batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"], pin_memory=(DEVICE == "cuda")
)

print(f"Source root: {source_root}")
print(f"Target root: {target_root}")
print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_subset)} | Val: {len(val_subset)} | Target (OOD): {len(target_dataset)}")
print("Train uses augmentation | Val uses deterministic transforms | Target is OOD only")


## 3. Model Architecture

### Stream 1: Log-Gabor Filter Bank + Phase Congruency

In [ ]:
# ============================================================
# LOG-GABOR FILTER BANK
# 24 filters (6 orientations x 4 scales), all non-learnable
# Computation entirely in frequency domain via FFT
# ============================================================

class LogGaborFilterBank(nn.Module):
    def __init__(
        self,
        image_size=(224, 224),
        num_scales=4,
        num_orientations=6,
        min_wavelength=3.0,
        mult=2.1,
        sigma_on_f=0.55,
        d_theta_on_sigma=1.2,
    ):
        super().__init__()
        self.num_scales = num_scales
        self.num_orientations = num_orientations
        self.image_size = image_size

        H, W = image_size
        # Match torch.fft.rfft2 layout: DC at [0, 0], unshifted vertical
        # frequencies, non-negative horizontal half-spectrum.
        u = torch.fft.fftfreq(H, dtype=torch.float32)
        v = torch.fft.rfftfreq(W, dtype=torch.float32)
        grid_u, grid_v = torch.meshgrid(u, v, indexing="ij")

        radius = torch.sqrt(grid_u**2 + grid_v**2).clamp(min=1e-8)
        theta = torch.atan2(grid_u, grid_v)
        d_theta = math.pi / num_orientations / d_theta_on_sigma

        filters = []
        sign_masks = []
        for s in range(num_scales):
            fo = 1.0 / (min_wavelength * (mult ** s))
            radial = torch.exp(
                -0.5 * (torch.log(radius / fo) ** 2) / (math.log(sigma_on_f) ** 2)
            )
            radial[0, 0] = 0.0  # zero DC
            for o in range(num_orientations):
                angle = o * math.pi / num_orientations
                ds = torch.sin(theta - angle)
                dc = torch.cos(theta - angle)
                angular = torch.exp(-0.5 * (torch.atan2(ds, dc).abs() ** 2) / (d_theta ** 2))
                filters.append(radial * angular)

                # Oriented Hilbert quadrature phase shift for this filter orientation.
                projection = grid_v * math.cos(angle) + grid_u * math.sin(angle)
                sign_masks.append(torch.sign(projection))

        self.register_buffer("filter_bank", torch.stack(filters, dim=0))  # (24, H, W//2+1)
        self.register_buffer("sign_mask", torch.stack(sign_masks, dim=0)) # (24, H, W//2+1)

    def forward(self, x):  # x: (B, 1, H, W)
        X_freq = torch.fft.rfft2(x)  # (B, 1, H, W//2+1)
        filtered = X_freq * self.filter_bank.unsqueeze(0)  # (B, 24, H, W//2+1)
        even = torch.fft.irfft2(filtered, s=self.image_size)  # (B, 24, H, W)
        hilbert = filtered * (-1j * self.sign_mask.unsqueeze(0))
        odd = torch.fft.irfft2(hilbert, s=self.image_size)  # (B, 24, H, W)
        return even, odd


print("LogGaborFilterBank defined (0 trainable parameters).")


In [ ]:
# ============================================================
# PHASE CONGRUENCY EXTRACTOR
# Produces: PC Magnitude, Phase Symmetry, Oriented Energy
# ============================================================

class PhaseCongruencyExtractor(nn.Module):
    def __init__(self, num_scales=4, num_orientations=6, noise_k=3.0, eps=1e-6, cutoff=0.5, gain=10.0):
        super().__init__()
        self.ns = num_scales
        self.no = num_orientations
        self.noise_k = noise_k
        self.eps = eps
        self.cutoff = cutoff
        self.gain = gain

    def _freq_spread_weight(self, amplitude):  # (B, no, ns, H, W)
        sum_amp = amplitude.sum(dim=2)
        max_amp = amplitude.max(dim=2).values
        width = (sum_amp / (max_amp + self.eps) - 1.0) / (self.ns - 1.0 + self.eps)
        return 1.0 / (1.0 + torch.exp(self.gain * (self.cutoff - width)))

    def forward(self, even, odd):  # each (B, ns*no, H, W)
        B, _, H, W = even.shape
        ns, no = self.ns, self.no

        even = even.view(B, ns, no, H, W).permute(0, 2, 1, 3, 4)  # (B, no, ns, H, W)
        odd = odd.view(B, ns, no, H, W).permute(0, 2, 1, 3, 4)

        amplitude = torch.sqrt(even**2 + odd**2 + self.eps)
        T = self.noise_k * amplitude[:, :, 0].flatten(2).median(dim=-1).values.unsqueeze(-1).unsqueeze(-1)
        W_s = self._freq_spread_weight(amplitude)
        sum_A = amplitude.sum(dim=2)

        # PC Magnitude
        energy = torch.sqrt(even.sum(2)**2 + odd.sum(2)**2 + self.eps)
        pc_orient = W_s * torch.clamp(energy - T, min=0) / (sum_A + self.eps)
        pc_mag = pc_orient.max(dim=1).values.unsqueeze(1)

        # Phase Symmetry
        sym_e = W_s * torch.clamp(even.abs().sum(2) - odd.abs().sum(2) - T, min=0)
        phase_sym = (sym_e / (sum_A + self.eps)).max(dim=1).values.unsqueeze(1)

        # Oriented Energy
        angles = torch.linspace(0, math.pi, no + 1, device=even.device)[:-1]
        oe_x = (energy * angles.cos().view(1, no, 1, 1)).sum(1, keepdim=True)
        oe_y = (energy * angles.sin().view(1, no, 1, 1)).sum(1, keepdim=True)
        oe = torch.sqrt(oe_x**2 + oe_y**2 + self.eps)

        def norm(t):
            flat = t.flatten(1)
            lo = flat.min(1).values.view(B, 1, 1, 1)
            hi = flat.max(1).values.view(B, 1, 1, 1)
            return (t - lo) / (hi - lo + self.eps)

        return {
            "pc_magnitude": norm(pc_mag),
            "phase_symmetry": norm(phase_sym),
            "oriented_energy": norm(oe),
        }


print("PhaseCongruencyExtractor defined.")

In [ ]:
# ============================================================
# PC ENCODER: 3-channel PC maps -> Structural Tokens (B, 49, D)
# ============================================================

class PCEncoder(nn.Module):
    def __init__(self, in_channels=3, mid_channels=64, fusion_dim=256, spatial_size=7):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, fusion_dim, 3, stride=2, padding=1),
            nn.BatchNorm2d(fusion_dim),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((spatial_size, spatial_size)),
        )

    def forward(self, pc_maps):  # (B, 3, H, W)
        features = self.encoder(pc_maps)  # (B, D, 7, 7)
        return features.flatten(2).transpose(1, 2)  # (B, 49, D)


print("PCEncoder defined.")

### Stream 2: ViT-B/16 Semantic Backbone

In [ ]:
# ============================================================
# SEMANTIC BACKBONE: ViT-B/16 via timm
# Outputs 196 patch tokens (14x14 grid) projected to fusion_dim
# ============================================================

class SemanticBackbone(nn.Module):
    def __init__(self, backbone_name="vit_base_patch16_224", fusion_dim=256, pretrained=True, freeze=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        backbone_dim = self.backbone.embed_dim  # 768 for vit_base

        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(backbone_dim, fusion_dim)
        self.norm = nn.LayerNorm(fusion_dim)

    def forward(self, x):  # (B, 3, 224, 224)
        tokens = self.backbone.forward_features(x)  # (B, 196+1, 768) or (B, 196, 768)
        if hasattr(self.backbone, "num_prefix_tokens") and self.backbone.num_prefix_tokens > 0:
            tokens = tokens[:, self.backbone.num_prefix_tokens:, :]  # drop CLS
        return self.norm(self.proj(tokens))  # (B, 196, fusion_dim)


print("SemanticBackbone (ViT-B/16) defined.")

### Stream 3: Illumination Normalisation

In [ ]:
# ============================================================
# ILLUMINATION-NORMALISED STREAM
# Shallow CNN on CLAHE-preprocessed images -> auxiliary vector
# ============================================================

class IlluminationNormStream(nn.Module):
    def __init__(self, in_channels=3, mid_channels=64, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_dim, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_dim),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x_clahe):  # (B, 3, H, W)
        return self.net(x_clahe).flatten(1)  # (B, out_dim)


print("IlluminationNormStream defined.")

### Cross-Attention Fusion + Full Model

In [ ]:
# ============================================================
# CROSS-ATTENTION FUSION
# Q = Structural Tokens (49), K/V = Semantic Tokens (196)
# Forces attention only where PC confirms structural boundaries
# ============================================================

class StructuralSemanticFusion(nn.Module):
    def __init__(self, fusion_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=fusion_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm_q = nn.LayerNorm(fusion_dim)
        self.norm_kv = nn.LayerNorm(fusion_dim)
        self.norm_out = nn.LayerNorm(fusion_dim)
        self.ffn = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.Dropout(dropout),
        )
        self.norm_ffn = nn.LayerNorm(fusion_dim)

    def forward(self, structural, semantic, return_attn=False):
        q = self.norm_q(structural)
        kv = self.norm_kv(semantic)
        attn_out, attn_w = self.cross_attn(q, kv, kv, need_weights=return_attn)
        attn_out = self.norm_out(attn_out + structural)  # residual
        ffn_out = self.ffn(attn_out)
        fused = self.norm_ffn(ffn_out + attn_out)  # (B, 49, D)
        return fused.mean(dim=1), attn_w  # (B, D), optional weights


print("StructuralSemanticFusion defined.")

In [ ]:
# ============================================================
# FULL PHASEPHYTO MODEL
# ============================================================

class PhasePhyto(nn.Module):
    def __init__(
        self,
        num_classes=38,
        backbone_name="vit_base_patch16_224",
        fusion_dim=256,
        pc_scales=4,
        pc_orientations=6,
        image_size=(224, 224),
        num_heads=4,
        dropout=0.1,
        pretrained=True,
        freeze_backbone=False,
    ):
        super().__init__()
        self.image_size = image_size

        # Stream 1: Phase Congruency
        self.filter_bank = LogGaborFilterBank(image_size, pc_scales, pc_orientations)
        self.pc_extractor = PhaseCongruencyExtractor(pc_scales, pc_orientations)
        self.pc_encoder = PCEncoder(in_channels=3, fusion_dim=fusion_dim)

        # Stream 2: ViT Semantic Backbone
        self.backbone = SemanticBackbone(backbone_name, fusion_dim, pretrained, freeze_backbone)

        # Stream 3: Illumination Normalisation
        self.illum_stream = IlluminationNormStream(out_dim=fusion_dim)

        # Fusion
        self.fusion = StructuralSemanticFusion(fusion_dim, num_heads, dropout)

        # Classifier: fused (D) + illumination (D) -> num_classes
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes),
        )

        # Luminance weights for RGB -> Gray
        self.register_buffer(
            "lum_weights",
            torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)
        )

    def forward(self, x_rgb, x_clahe=None, return_maps=False, return_attn=False):
        if x_clahe is None:
            x_clahe = x_rgb

        # Stream 1: Phase Congruency
        gray = (x_rgb * self.lum_weights).sum(dim=1, keepdim=True)  # (B, 1, H, W)
        even, odd = self.filter_bank(gray)
        pc = self.pc_extractor(even, odd)
        pc_input = torch.cat([pc["pc_magnitude"], pc["phase_symmetry"], pc["oriented_energy"]], dim=1)
        structural_tokens = self.pc_encoder(pc_input)  # (B, 49, D)

        # Stream 2: Semantic Backbone
        semantic_tokens = self.backbone(x_rgb)  # (B, 196, D)

        # Stream 3: Illumination
        illum = self.illum_stream(x_clahe)  # (B, D)

        # Fusion
        fused, attn_w = self.fusion(structural_tokens, semantic_tokens, return_attn)  # (B, D)

        # Classify
        logits = self.classifier(torch.cat([fused, illum], dim=1))  # (B, num_classes)

        out = {"logits": logits}
        if return_maps:
            out.update(pc)
            out["pc_input"] = pc_input
            out["fused"] = fused
        if return_attn and attn_w is not None:
            out["attn_weights"] = attn_w
        return out

    def count_parameters(self):
        counts = {}
        for name, module in [
            ("filter_bank", self.filter_bank), ("pc_extractor", self.pc_extractor),
            ("pc_encoder", self.pc_encoder), ("backbone", self.backbone),
            ("illum_stream", self.illum_stream), ("fusion", self.fusion),
            ("classifier", self.classifier),
        ]:
            counts[name] = sum(p.numel() for p in module.parameters() if p.requires_grad)
        counts["total"] = sum(counts.values())
        return counts


# Build model
model = PhasePhyto(
    num_classes=NUM_CLASSES,
    backbone_name=CONFIG["backbone_name"],
    fusion_dim=CONFIG["fusion_dim"],
    pc_scales=CONFIG["pc_scales"],
    pc_orientations=CONFIG["pc_orientations"],
    image_size=(CONFIG["image_size"], CONFIG["image_size"]),
    num_heads=CONFIG["num_heads"],
    dropout=CONFIG["dropout"],
    pretrained=True,
).to(DEVICE)

# Parameter report
counts = model.count_parameters()
print("\n=== Parameter Counts ===")
for k, v in counts.items():
    print(f"  {k:20s}: {v:>12,}")
print(f"\n  Filter bank + PC extractor: 0 trainable (physics-based)")
print(f"  Fusion module:              ~{counts['fusion']:,} params (cross-attention)")

## 4. Verify Forward + Backward Pass

In [ ]:
# Quick sanity check
print("Testing forward pass...")
x_test = torch.randn(2, 3, 224, 224, device=DEVICE)
with torch.no_grad():
    out_test = model(x_test, return_maps=True, return_attn=True)

print(f"  Logits shape:          {out_test['logits'].shape}")  # (2, NUM_CLASSES)
print(f"  PC Magnitude shape:    {out_test['pc_magnitude'].shape}")  # (2, 1, 224, 224)
print(f"  Phase Symmetry shape:  {out_test['phase_symmetry'].shape}")
print(f"  Oriented Energy shape: {out_test['oriented_energy'].shape}")
print(f"  Attention weights:     {out_test['attn_weights'].shape}")  # (2, 49, 196)

print("\nTesting backward pass...")
x_grad = torch.randn(2, 3, 224, 224, device=DEVICE)
out_grad = model(x_grad)
loss_test = out_grad["logits"].sum()
loss_test.backward()

n_grad = sum(1 for n, p in model.named_parameters() if p.requires_grad and p.grad is not None)
n_req = sum(1 for p in model.parameters() if p.requires_grad)
print(f"  Gradients computed: {n_grad}/{n_req} learnable parameters")
model.zero_grad()
print("  PASS")

In [ ]:
# ============================================================
# VERIFY AMPLITUDE INVARIANCE: PC(image) == PC(image * k)
# This is the core physics property that makes PhasePhyto work
# ============================================================

print("Verifying amplitude invariance of Phase Congruency...")
fb = model.filter_bank
pc_ext = model.pc_extractor

test_img = torch.rand(1, 1, 224, 224, device=DEVICE) * 0.5 + 0.1
even_ref, odd_ref = fb(test_img)
pc_ref = pc_ext(even_ref, odd_ref)

for k in [0.1, 0.5, 2.0, 5.0, 10.0]:
    even_k, odd_k = fb(test_img * k)
    pc_k = pc_ext(even_k, odd_k)
    diffs = {key: (pc_ref[key] - pc_k[key]).abs().max().item() for key in pc_ref}
    status = "PASS" if all(d < 0.005 for d in diffs.values()) else "FAIL"
    print(f"  k={k:5.1f}: max_diff = {max(diffs.values()):.6f} [{status}]")

print("\nAmplitude invariance verified: lighting changes don't affect structural features.")

## 5. Training

In [ ]:
# ============================================================
# FOCAL LOSS (for class imbalance)
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()


# ============================================================
# TRAINING LOOP
# ============================================================

criterion = FocalLoss(gamma=2.0)
optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=max(CONFIG["epochs"] - CONFIG["warmup_epochs"], 1))
scaler = GradScaler(DEVICE, enabled=(DEVICE == "cuda"))


def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        if len(batch) == 3:
            rgb, clahe, labels = [b.to(device) for b in batch]
        else:
            rgb, labels = batch[0].to(device), batch[1].to(device)
            clahe = None

        optimizer.zero_grad()
        with autocast(device_type="cuda", enabled=(device == "cuda")):
            out = model(rgb, x_clahe=clahe)
            loss = criterion(out["logits"], labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        correct += (out["logits"].argmax(1) == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.3f}")

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, labels_all = 0.0, [], []

    for batch in loader:
        if len(batch) == 3:
            rgb, clahe, labels = [b.to(device) for b in batch]
        else:
            rgb, labels = batch[0].to(device), batch[1].to(device)
            clahe = None

        out = model(rgb, x_clahe=clahe)
        total_loss += criterion(out["logits"], labels).item() * labels.size(0)
        preds.extend(out["logits"].argmax(1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())

    acc = sum(p == l for p, l in zip(preds, labels_all)) / len(labels_all)
    f1 = f1_score(labels_all, preds, average="macro", zero_division=0)
    return total_loss / len(labels_all), acc, f1


print("Training functions defined.")

In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# ============================================================

best_f1 = 0.0
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

print(f"\nStarting training: {CONFIG['epochs']} epochs, batch_size={CONFIG['batch_size']}")
print(f"Backbone: {CONFIG['backbone_name']} | Fusion dim: {CONFIG['fusion_dim']}")
print(f"{'='*70}\n")

for epoch in range(CONFIG["epochs"]):
    # Warmup LR
    if epoch < CONFIG["warmup_epochs"]:
        factor = (epoch + 1) / CONFIG["warmup_epochs"]
        for pg in optimizer.param_groups:
            pg["lr"] = CONFIG["lr"] * factor

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, epoch)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, DEVICE)

    if epoch >= CONFIG["warmup_epochs"]:
        scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:3d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f} | lr={lr:.2e}")

    # Checkpoint best
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_f1": best_f1,
        }, CHECKPOINT_DIR / "best_phasephyto.pt")
        print(f"  -> Best model saved to {CHECKPOINT_DIR / 'best_phasephyto.pt'} (F1={best_f1:.4f})")
    else:
        patience_counter += 1

    if CONFIG["patience"] > 0 and patience_counter >= CONFIG["patience"]:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best validation F1: {best_f1:.4f}")


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], label="Train", linewidth=2)
axes[0].plot(history["val_loss"], label="Val", linewidth=2)
axes[0].set_title("Loss", fontsize=14)
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["train_acc"], label="Train", linewidth=2)
axes[1].plot(history["val_acc"], label="Val", linewidth=2)
axes[1].set_title("Accuracy", fontsize=14)
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history["val_f1"], label="Val F1 (macro)", linewidth=2, color="green")
axes[2].axhline(y=best_f1, color="red", linestyle="--", alpha=0.7, label=f"Best: {best_f1:.4f}")
axes[2].set_title("Validation F1", fontsize=14)
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "training_curves.png", dpi=150)
plt.show()


## 7. Domain Shift Evaluation: PlantVillage -> PlantDoc

In [ ]:
# Load best model
ckpt = torch.load(CHECKPOINT_DIR / "best_phasephyto.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['epoch']+1} (F1={ckpt['val_f1']:.4f})")

# Evaluate on source (PlantVillage val) and target (PlantDoc)
print("\n" + "="*60)
print("DOMAIN SHIFT EVALUATION")
print("="*60)

_, source_acc, source_f1 = evaluate(model, val_loader, criterion, DEVICE)
_, target_acc, target_f1 = evaluate(model, target_loader, criterion, DEVICE)

delta_acc = target_acc - source_acc
delta_f1 = target_f1 - source_f1

print(f"\n{'Metric':<20} {'Source (PV)':<15} {'Target (PD)':<15} {'Delta':<15}")
print("-" * 65)
print(f"{'Accuracy':<20} {source_acc:<15.4f} {target_acc:<15.4f} {delta_acc:<+15.4f}")
print(f"{'F1 (macro)':<20} {source_f1:<15.4f} {target_f1:<15.4f} {delta_f1:<+15.4f}")
print(f"\nAccuracy drop: {abs(delta_acc)*100:.1f}%")
print(f"Target: delta < 5% | Achieved: {abs(delta_acc)*100:.1f}%")

# Persist domain-shift metrics immediately.
domain_shift_results = {
    "source_acc": source_acc,
    "source_f1": source_f1,
    "target_acc": target_acc,
    "target_f1": target_f1,
    "delta_acc": delta_acc,
    "delta_f1": delta_f1,
}
with open(RESULTS_DIR / "phasephyto_domain_shift.json", "w") as f:
    json.dump(domain_shift_results, f, indent=2)
print(f"Domain-shift metrics saved to {RESULTS_DIR / 'phasephyto_domain_shift.json'}")


In [ ]:
# Detailed classification report on target domain
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        if len(batch) == 3:
            rgb, clahe, lbl = [b.to(device) for b in batch]
        else:
            rgb, lbl = batch[0].to(device), batch[1].to(device)
            clahe = None
        out = model(rgb, x_clahe=clahe)
        preds.extend(out["logits"].argmax(1).cpu().tolist())
        labels.extend(lbl.cpu().tolist())
    return labels, preds

true_labels, pred_labels = get_predictions(model, target_loader, DEVICE)
print("\nTarget Domain (PlantDoc) Classification Report:")
target_labels = list(range(NUM_CLASSES))
report_text = classification_report(
    true_labels,
    pred_labels,
    labels=target_labels,
    target_names=train_dataset.classes,
    zero_division=0,
)
print(report_text)
with open(RESULTS_DIR / "target_classification_report.txt", "w") as f:
    f.write(report_text)


## 8. Visualisation: Phase Congruency Maps & Attention

In [ ]:
# Visualise PC maps for sample images
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1)


def visualize_sample(model, dataset, idx, device):
    """Visualise a single sample: Original | PC Mag | Phase Sym | Oriented Energy."""
    model.eval()
    sample = dataset[idx]
    if len(sample) == 3:
        rgb, clahe, label = sample
    else:
        rgb, label = sample
        clahe = rgb

    with torch.no_grad():
        out = model(
            rgb.unsqueeze(0).to(device),
            x_clahe=clahe.unsqueeze(0).to(device),
            return_maps=True, return_attn=True
        )

    pred = out["logits"].argmax(1).item()
    conf = torch.softmax(out["logits"], dim=1).max().item()

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    # Original image
    img_display = denormalize(rgb).permute(1, 2, 0).numpy()
    axes[0].imshow(img_display)
    class_name = dataset.dataset.classes[label] if hasattr(dataset, 'dataset') else str(label)
    pred_name = dataset.dataset.classes[pred] if hasattr(dataset, 'dataset') else str(pred)
    axes[0].set_title(f"Input\nTrue: {class_name}\nPred: {pred_name} ({conf:.1%})", fontsize=10)

    # PC Maps
    for i, (key, cmap, title) in enumerate([
        ("pc_magnitude", "hot", "PC Magnitude\n(Edges & Boundaries)"),
        ("phase_symmetry", "magma", "Phase Symmetry\n(Symmetric Structures)"),
        ("oriented_energy", "viridis", "Oriented Energy\n(Directional Textures)"),
    ]):
        axes[i+1].imshow(out[key][0, 0].cpu().numpy(), cmap=cmap)
        axes[i+1].set_title(title, fontsize=10)

    # Attention map (reshaped from 49 tokens to 7x7)
    attn = out["attn_weights"][0].cpu()  # (49, 196)
    attn_map = attn.mean(dim=0).view(14, 14).numpy()  # average across Q, reshape K to 14x14
    axes[4].imshow(img_display)
    attn_resized = cv2.resize(attn_map, (224, 224), interpolation=cv2.INTER_LINEAR)
    axes[4].imshow(attn_resized, cmap="jet", alpha=0.5)
    axes[4].set_title("Cross-Attention\nOverlay", fontsize=10)

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    return fig


# Visualise 3 samples from target domain
print("Visualising PhasePhyto analysis on target domain samples...\n")
for i in range(min(3, len(target_dataset))):
    fig = visualize_sample(model, target_dataset, i, DEVICE)
    plt.savefig(PLOTS_DIR / f"analysis_sample_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# ============================================================
# ILLUMINATION INVARIANCE DEMO
# Show that PC maps remain stable across brightness variations
# ============================================================

print("Demonstrating illumination invariance...")
model.eval()

# Take first image from target set
sample = target_dataset[0]
rgb = sample[0] if isinstance(sample[0], torch.Tensor) else sample
rgb = rgb.unsqueeze(0).to(DEVICE)

brightness_factors = [0.3, 0.5, 1.0, 1.5, 2.5]
fig, axes = plt.subplots(3, len(brightness_factors), figsize=(20, 12))

for j, k in enumerate(brightness_factors):
    # Scale brightness in image space (before normalisation this simulates lighting change)
    scaled = rgb * k
    with torch.no_grad():
        out = model(scaled, return_maps=True)

    # Row 1: Scaled image
    img_show = denormalize(scaled[0].cpu()).permute(1, 2, 0).clamp(0, 1).numpy()
    axes[0, j].imshow(img_show)
    axes[0, j].set_title(f"Brightness x{k}", fontsize=11)

    # Row 2: PC Magnitude
    axes[1, j].imshow(out["pc_magnitude"][0, 0].cpu().numpy(), cmap="hot", vmin=0, vmax=1)
    if j == 0:
        axes[1, j].set_ylabel("PC Magnitude", fontsize=12)

    # Row 3: Oriented Energy
    axes[2, j].imshow(out["oriented_energy"][0, 0].cpu().numpy(), cmap="viridis", vmin=0, vmax=1)
    if j == 0:
        axes[2, j].set_ylabel("Oriented Energy", fontsize=12)

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Phase Congruency is Invariant to Illumination Changes\n"
             "PC(image) = PC(image * k) for k > 0 -- structural maps stay stable",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "illumination_invariance.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Baseline Comparison: Standard ViT vs PhasePhyto

In [ ]:
# ============================================================
# BASELINE: Standard ViT-B/16 (no phase congruency)
# Same backbone, same training config, no PC stream
# ============================================================

class BaselineViT(nn.Module):
    """Standard ViT-B/16 classifier without Phase Congruency."""
    def __init__(self, num_classes, backbone_name="vit_base_patch16_224", pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=num_classes)

    def forward(self, x_rgb, x_clahe=None, **kwargs):
        logits = self.backbone(x_rgb)
        return {"logits": logits}


# Train baseline (same config)
print("Training Baseline ViT for comparison...")
baseline = BaselineViT(NUM_CLASSES).to(DEVICE)
baseline_opt = AdamW(baseline.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
baseline_sched = CosineAnnealingWarmRestarts(baseline_opt, T_0=max(CONFIG["epochs"] - CONFIG["warmup_epochs"], 1))
baseline_scaler = GradScaler(DEVICE, enabled=(DEVICE == "cuda"))

baseline_best_f1 = 0.0
for epoch in range(CONFIG["epochs"]):
    if epoch < CONFIG["warmup_epochs"]:
        for pg in baseline_opt.param_groups:
            pg["lr"] = CONFIG["lr"] * (epoch + 1) / CONFIG["warmup_epochs"]

    t_loss, t_acc = train_epoch(baseline, train_loader, criterion, baseline_opt, baseline_scaler, DEVICE, epoch)
    v_loss, v_acc, v_f1 = evaluate(baseline, val_loader, criterion, DEVICE)

    if epoch >= CONFIG["warmup_epochs"]:
        baseline_sched.step()

    print(f"  Baseline Epoch {epoch+1}: val_acc={v_acc:.4f} val_f1={v_f1:.4f}")
    if v_f1 > baseline_best_f1:
        baseline_best_f1 = v_f1
        torch.save({"model_state_dict": baseline.state_dict(), "val_f1": v_f1}, CHECKPOINT_DIR / "baseline_vit.pt")

print(f"\nBaseline best val F1: {baseline_best_f1:.4f}")


In [ ]:
# ============================================================
# COMPARISON TABLE
# ============================================================

# Load best baseline
baseline_ckpt = torch.load(CHECKPOINT_DIR / "baseline_vit.pt", map_location=DEVICE, weights_only=True)
baseline.load_state_dict(baseline_ckpt["model_state_dict"])

# Evaluate both
_, bl_source_acc, bl_source_f1 = evaluate(baseline, val_loader, criterion, DEVICE)
_, bl_target_acc, bl_target_f1 = evaluate(baseline, target_loader, criterion, DEVICE)

_, pp_source_acc, pp_source_f1 = evaluate(model, val_loader, criterion, DEVICE)
_, pp_target_acc, pp_target_f1 = evaluate(model, target_loader, criterion, DEVICE)

print("\n" + "=" * 80)
print("COMPARISON: Standard ViT-B/16 vs PhasePhyto (ViT-B/16 + Phase Congruency)")
print("=" * 80)
print(f"\n{'Model':<25} {'Source Acc':<12} {'Source F1':<12} {'Target Acc':<12} {'Target F1':<12} {'Delta Acc':<12}")
print("-" * 85)
print(f"{'ViT-B/16 (baseline)':<25} {bl_source_acc:<12.4f} {bl_source_f1:<12.4f} {bl_target_acc:<12.4f} {bl_target_f1:<12.4f} {(bl_target_acc-bl_source_acc):<+12.4f}")
print(f"{'PhasePhyto (ours)':<25} {pp_source_acc:<12.4f} {pp_source_f1:<12.4f} {pp_target_acc:<12.4f} {pp_target_f1:<12.4f} {(pp_target_acc-pp_source_acc):<+12.4f}")
print(f"\nPhasePhyto improvement on target: {(pp_target_f1 - bl_target_f1)*100:+.1f}% F1")
print(f"PhasePhyto domain gap reduction:  {abs(pp_target_acc-pp_source_acc)*100:.1f}% vs {abs(bl_target_acc-bl_source_acc)*100:.1f}%")


## 10. Confusion Matrix Comparison

In [ ]:
import seaborn as sns

# Get predictions for both models on target
bl_true, bl_pred = get_predictions(baseline, target_loader, DEVICE)
pp_true, pp_pred = get_predictions(model, target_loader, DEVICE)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

target_labels = list(range(NUM_CLASSES))
cm_bl = confusion_matrix(bl_true, bl_pred, labels=target_labels)
cm_pp = confusion_matrix(pp_true, pp_pred, labels=target_labels)

sns.heatmap(cm_bl, annot=True, fmt="d", cmap="Blues", ax=ax1, cbar=False)
ax1.set_title(f"Baseline ViT-B/16\nTarget Acc: {bl_target_acc:.3f} | F1: {bl_target_f1:.3f}", fontsize=13)
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True")

sns.heatmap(cm_pp, annot=True, fmt="d", cmap="Greens", ax=ax2, cbar=False)
ax2.set_title(f"PhasePhyto\nTarget Acc: {pp_target_acc:.3f} | F1: {pp_target_f1:.3f}", fontsize=13)
ax2.set_xlabel("Predicted")
ax2.set_ylabel("True")

plt.suptitle("Confusion Matrices on Target Domain (PlantDoc)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "confusion_matrices.png", dpi=150)
plt.show()


## 11. Export & Save

In [ ]:
# Save final results + artifact manifest
import json
from pathlib import Path

results = {
    "config": CONFIG,
    "run_dir": str(RUN_DIR),
    "baseline_vit": {
        "source_acc": bl_source_acc, "source_f1": bl_source_f1,
        "target_acc": bl_target_acc, "target_f1": bl_target_f1,
        "delta_acc": bl_target_acc - bl_source_acc,
    },
    "phasephyto": {
        "source_acc": pp_source_acc, "source_f1": pp_source_f1,
        "target_acc": pp_target_acc, "target_f1": pp_target_f1,
        "delta_acc": pp_target_acc - pp_source_acc,
    },
    "parameter_counts": counts,
    "training_history": history,
}

results_path = RESULTS_DIR / "phasephyto_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2, default=str)

manifest = {
    "run_name": RUN_NAME,
    "storage_backend": backend,
    "run_dir": str(RUN_DIR),
    "data": {
        "source_root": str(source_root),
        "target_root": str(target_root),
    },
    "config": CONFIG,
    "artifacts": {
        "phasephyto_checkpoint": str(CHECKPOINT_DIR / "best_phasephyto.pt"),
        "baseline_checkpoint": str(CHECKPOINT_DIR / "baseline_vit.pt"),
        "results_json": str(results_path),
        "domain_shift_json": str(RESULTS_DIR / "phasephyto_domain_shift.json"),
        "target_classification_report": str(RESULTS_DIR / "target_classification_report.txt"),
        "training_curves": str(PLOTS_DIR / "training_curves.png"),
        "illumination_invariance": str(PLOTS_DIR / "illumination_invariance.png"),
        "confusion_matrices": str(PLOTS_DIR / "confusion_matrices.png"),
        "analysis_samples_glob": str(PLOTS_DIR / "analysis_sample_*.png"),
    },
}
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Results saved to {results_path}")
print(f"Manifest saved to {MANIFEST_PATH}")
print(f"Best model saved to {CHECKPOINT_DIR / 'best_phasephyto.pt'}")
print(f"Baseline model saved to {CHECKPOINT_DIR / 'baseline_vit.pt'}")
print(f"Plots saved under {PLOTS_DIR}")

# Optional fallback downloads. Drive storage is preferred for long-running jobs.
if CONFIG.get("download_outputs_at_end", False):
    try:
        from google.colab import files
        files.download(str(results_path))
        files.download(str(MANIFEST_PATH))
        files.download(str(CHECKPOINT_DIR / "best_phasephyto.pt"))
        files.download(str(PLOTS_DIR / "training_curves.png"))
        files.download(str(PLOTS_DIR / "illumination_invariance.png"))
    except ImportError:
        print("Not running in Colab -- files saved to the run directory.")


---

## Summary

This notebook implements the complete **PhasePhyto** pipeline:

| Component | Description | Trainable Params |
|-----------|-------------|------------------|
| Log-Gabor Filter Bank | 24 filters (6 orient x 4 scale), FFT-based | **0** |
| PC Extractor | Magnitude, Symmetry, Oriented Energy maps | **0** |
| PC Encoder | 2-layer CNN -> 49 structural tokens | ~50K |
| ViT-B/16 Backbone | 196 semantic tokens (14x14 patch grid) | ~86M |
| Illumination Stream | CLAHE + shallow CNN -> auxiliary vector | ~50K |
| Cross-Attention Fusion | Q=structural, K/V=semantic | ~331K |
| Classifier | MLP head | ~66K |

**Key Result**: Phase Congruency provides mathematical illumination invariance:
`PC(image) = PC(image * k)` for any positive scalar `k`.

This tests the physics stream for brightness-scaling invariance. End-to-end
lab-to-field gains should be interpreted only after comparing PhasePhyto against
the baseline on the same source/target split.

---

**References**:
- Kovesi, P. (1999). Image Features from Phase Congruency. *Videre*.
- Vidyarthi et al. (2024). PhaseHisto.
- PlantVillage: Hughes & Salathé (2015).
- PlantDoc: Singh et al. (2020).
- XyloTron: Hermanson & Wiedenhoeft (2011).
- Dosovitskiy et al. (2020). An Image is Worth 16x16 Words (ViT).